# 01 Dataset Preparation

This notebook prepares the district-level Urban Heat Island dataset by combining in-situ meteorological observations, station information, Landsat-derived variables, UHI intensity labels, stratified train-test split, and SMOTE balancing.

**Project:** Comparative Study of Machine Learning Models for District-Level Urban Heat Island Classification in Taiwan

**Author:** Zalfa Afifah Zahra

## Ground-truth and station data preparation

Combine, pivot, and organize Taiwan EPA monitoring data and station information.

In [ ]:
# ============================================================
# COMBINE ALL CSV FILES FROM GROUND TRUTH ZIP
# ============================================================

import os
import glob
import zipfile
import pandas as pd

# ============================================================
# 1. ZIP FILE PATH
# ============================================================

zip_path = "/content/ground_truth.zip"

# ============================================================
# 2. EXTRACT ZIP
# ============================================================

extract_dir = "/content/ground_truth_extracted"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("="*60)
print("ZIP EXTRACTION COMPLETED")
print("="*60)
print(f"ZIP File     : {zip_path}")
print(f"Extracted To : {extract_dir}")

# ============================================================
# 3. FIND ALL CSV FILES
# ============================================================

csv_files = glob.glob(
    os.path.join(extract_dir, "**", "*.csv"),
    recursive=True
)

print("\n" + "="*60)
print("CSV FILE DISCOVERY")
print("="*60)
print(f"Total CSV Files Found: {len(csv_files)}")

# ============================================================
# 4. READ ALL CSV FILES
# ============================================================

all_dfs = []

encodings = [
    "utf-8",
    "utf-8-sig",
    "big5",
    "cp950",
    "latin1"
]

for file in csv_files:

    loaded = False

    for enc in encodings:

        try:
            df = pd.read_csv(
                file,
                encoding=enc,
                low_memory=False
            )

            df["source_file"] = os.path.basename(file)

            all_dfs.append(df)

            print(f"✓ {os.path.basename(file)} ({enc})")

            loaded = True
            break

        except:
            pass

    if not loaded:
        print(f"✗ Failed: {os.path.basename(file)}")

# ============================================================
# 5. COMBINE DATASETS
# ============================================================

combined_df = pd.concat(
    all_dfs,
    ignore_index=True
)

# ============================================================
# 6. FIX DATE COLUMN
# ============================================================

if "monitordate" in combined_df.columns:

    combined_df["monitordate"] = pd.to_datetime(
        combined_df["monitordate"],
        errors="coerce",
        dayfirst=True
    )

# ============================================================
# 7. DATASET SUMMARY
# ============================================================

print("\n" + "="*60)
print("COMBINED DATASET SUMMARY")
print("="*60)

print(f"Rows    : {combined_df.shape[0]:,}")
print(f"Columns : {combined_df.shape[1]}")

print("\nColumn Names:")
print(combined_df.columns.tolist())

print("\nMissing Values:")
print(combined_df.isnull().sum().head(20))

print("\nFirst 5 Rows:")
display(combined_df.head())

# ============================================================
# 8. SAVE OUTPUT
# ============================================================

output_file = "/content/ground_truth_combined.csv"

combined_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "="*60)
print("EXPORT COMPLETED")
print("="*60)
print(f"Saved To: {output_file}")

# ============================================================
# 9. CHECK FOR BROKEN CHARACTERS
# ============================================================

print("\nChecking for broken text (????) ...")

for col in combined_df.columns:

    if combined_df[col].dtype == object:

        sample = combined_df[col].astype(str)

        if sample.str.contains(r"\?{2,}", regex=True).any():

            print(f"Possible encoding issue in column: {col}")

print("\nDone.")

In [ ]:
# ============================================================
# PIVOT AIR QUALITY PARAMETERS
# ============================================================

import pandas as pd

# Parameter yang ingin dipertahankan
selected_items = [
    'PM10',
    'O3',
    'CO',
    'SO2',
    'WS_HR',
    'RH',
    'PM2.5',
    'CH4',
    'AMB_TEMP',
    'WIND_SPEED',
    'NMHC',
    'THC',
    'NO2',
    'NO',
    'NOx',
    'CO2'
]

# Filter parameter yang dibutuhkan
df_filtered = combined_df[
    combined_df['itemengname'].isin(selected_items)
].copy()

# Pastikan concentration numerik
df_filtered['concentration'] = pd.to_numeric(
    df_filtered['concentration'],
    errors='coerce'
)

# Pivot table
pivot_df = df_filtered.pivot_table(
    index=['siteid', 'sitename', 'monitordate'],
    columns='itemengname',
    values='concentration',
    aggfunc='mean'
).reset_index()

# Hilangkan nama level kolom
pivot_df.columns.name = None

# Urutan kolom
desired_order = [
    'siteid',
    'sitename',
    'monitordate',
    'PM10',
    'O3',
    'CO',
    'SO2',
    'WS_HR',
    'RH',
    'PM2.5',
    'CH4',
    'AMB_TEMP',
    'WIND_SPEED',
    'NMHC',
    'THC',
    'NO2',
    'NO',
    'NOx',
    'CO2'
]

existing_cols = [c for c in desired_order if c in pivot_df.columns]
pivot_df = pivot_df[existing_cols]

print("="*60)
print("PIVOT COMPLETED")
print("="*60)
print("Shape:", pivot_df.shape)

display(pivot_df.head())

# Save
pivot_output = "/content/ground_truth_pivot.csv"

pivot_df.to_csv(
    pivot_output,
    index=False,
    encoding='utf-8-sig'
)

print(f"\nSaved: {pivot_output}")

In [ ]:
# ============================================================
# LOAD STATION SHAPEFILE
# ============================================================

!pip -q install geopandas pyogrio

import geopandas as gpd

shp_file = "/content/station_location/空氣品質監測站位置圖.shp"

gdf = gpd.read_file(shp_file)

print("="*60)
print("SHAPEFILE LOADED")
print("="*60)

print("CRS:")
print(gdf.crs)

print("\nColumns:")
print(gdf.columns.tolist())

display(gdf.head())

# ============================================================
# ADD LATITUDE & LONGITUDE TO PIVOT DATASET
# ============================================================

# Ambil kolom yang diperlukan dari shapefile
station_lookup = gdf[
    [
        "Stcode",
        "SiteName",
        "SiteEngNam",
        "TWD97_Lon",
        "TWD97_Lat"
    ]
].copy()

# Rename agar konsisten
station_lookup = station_lookup.rename(
    columns={
        "Stcode": "station_code",
        "SiteName": "station_name_tw",
        "SiteEngNam": "sitename",
        "TWD97_Lon": "lon",
        "TWD97_Lat": "lat"
    }
)

# Hilangkan duplikat
station_lookup = station_lookup.drop_duplicates(
    subset=["sitename"]
)

print("Station lookup shape:", station_lookup.shape)

# ============================================================
# MERGE
# ============================================================

pivot_geo = pivot_df.merge(
    station_lookup,
    on="sitename",
    how="left"
)

print("\nMerged shape:", pivot_geo.shape)

# ============================================================
# CHECK MISSING COORDINATES
# ============================================================

missing = pivot_geo[pivot_geo["lat"].isna()]

print("\nStations without coordinates:")
print(missing["sitename"].dropna().unique())

print("\nTotal missing rows:", len(missing))

# ============================================================
# REORDER COLUMNS
# ============================================================

cols = [
    "siteid",
    "sitename",
    "station_code",
    "station_name_tw",
    "lat",
    "lon",
    "monitordate"
]

other_cols = [c for c in pivot_geo.columns if c not in cols]

pivot_geo = pivot_geo[
    cols + other_cols
]

# ============================================================
# SAVE
# ============================================================

output_file = "/content/ground_truth_pivot_with_coordinates.csv"

pivot_geo.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(output_file)

display(pivot_geo.head())

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

import pandas as pd
import numpy as np

ground_truth = pd.read_csv(
    "/content/ground_truth_pivot_with_coordinates.csv"
)

landsat = pd.read_csv(
    "/content/Taiwan_UHI_Landsat89_2025_2026.csv"
)

print("=" * 60)
print("DATA LOADED")
print("=" * 60)

print("Ground Truth Shape :", ground_truth.shape)
print("Landsat Shape      :", landsat.shape)


# ============================================================
# CHECK COLUMNS
# ============================================================

print("\nGround Truth Columns")
print(ground_truth.columns.tolist())

print("\nLandsat Columns")
print(landsat.columns.tolist())


# ============================================================
# DATE CONVERSION
# ============================================================

ground_truth["monitordate"] = pd.to_datetime(
    ground_truth["monitordate"],
    errors="coerce"
)

landsat["image_date"] = pd.to_datetime(
    landsat["image_date"],
    errors="coerce"
)

print("\nDate Ranges")

print(
    "Ground Truth :",
    ground_truth["monitordate"].min(),
    "->",
    ground_truth["monitordate"].max()
)

print(
    "Landsat :",
    landsat["image_date"].min(),
    "->",
    landsat["image_date"].max()
)


# ============================================================
# STANDARDIZE SITE NAMES
# ============================================================

ground_truth["sitename"] = (
    ground_truth["sitename"]
    .astype(str)
    .str.strip()
)

landsat["sitename"] = (
    landsat["sitename"]
    .astype(str)
    .str.strip()
)


# ============================================================
# SORT DATA
# ============================================================

ground_truth = ground_truth.sort_values(
    ["sitename", "monitordate"]
)

landsat = landsat.sort_values(
    ["sitename", "image_date"]
)


# ============================================================
# TEMPORAL MATCHING ±8 DAYS
# ============================================================

merged_list = []

all_sites = sorted(
    ground_truth["sitename"].unique()
)

print("\nTotal Sites:", len(all_sites))

for site in all_sites:

    gt_site = ground_truth[
        ground_truth["sitename"] == site
    ].copy()

    ls_site = landsat[
        landsat["sitename"] == site
    ].copy()

    if len(ls_site) == 0:

        print(
            f"No Landsat data for {site}"
        )

        continue

    merged_site = pd.merge_asof(

        gt_site.sort_values(
            "monitordate"
        ),

        ls_site.sort_values(
            "image_date"
        ),

        left_on="monitordate",

        right_on="image_date",

        direction="nearest",

        tolerance=pd.Timedelta(
            days=8
        )

    )

    merged_list.append(
        merged_site
    )


merged = pd.concat(
    merged_list,
    ignore_index=True
)

print("\n" + "=" * 60)
print("TEMPORAL MATCHING COMPLETED")
print("=" * 60)

print("Rows :", len(merged))
print("Cols :", len(merged.columns))


# ============================================================
# DATE DIFFERENCE
# ============================================================

merged["date_diff_days"] = (

    merged["image_date"]
    - merged["monitordate"]

).dt.days

print("\nDate Difference Statistics")

print(
    merged["date_diff_days"]
    .describe()
)


# ============================================================
# CHECK MISSING LANDSAT FEATURES
# ============================================================

landsat_features = [

    "LST_C",
    "NDVI",
    "NDBI",
    "NDWI",
    "MNDWI",
    "BSI",
    "EVI",
    "SAVI",
    "Albedo"

]

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

for col in landsat_features:

    missing = merged[col].isna().sum()

    print(
        f"{col:<10} : {missing:,}"
    )


# ============================================================
# MATCHING SUCCESS RATE
# ============================================================

valid = merged["LST_C"].notna().sum()

total = len(merged)

print("\n" + "=" * 60)
print("MATCHING SUCCESS")
print("=" * 60)

print(
    f"Matched Samples : {valid:,}"
)

print(
    f"Total Samples   : {total:,}"
)

print(
    f"Success Rate    : {100*valid/total:.2f}%"
)


# ============================================================
# REMOVE ROWS WITHOUT LANDSAT DATA
# ============================================================

merged_clean = merged.dropna(
    subset=landsat_features
).reset_index(drop=True)

print("\n" + "=" * 60)
print("AFTER REMOVING NaN")
print("=" * 60)

print(
    "Rows:",
    len(merged_clean)
)

print(
    "Removed:",
    len(merged) - len(merged_clean)
)


# ============================================================
# PREVIEW
# ============================================================

display(
    merged_clean.head()
)


# ============================================================
# SAVE CLEAN DATASET
# ============================================================

output_file = (
    "/content/"
    "ground_truth_uhi_landsat.csv"
)

merged_clean.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 60)
print("EXPORT COMPLETED")
print("=" * 60)

print(
    "Saved to:"
)

print(output_file)

## UHI label generation

Calculate UHI intensity from Landsat-derived LST and assign five standard deviation-based UHI classes.

In [ ]:
# ==========================================================
# UHI CLASSIFICATION (5 CLASSES)
# Based on Mean ± Standard Deviation of LST
# ==========================================================

import pandas as pd
import numpy as np

# ==========================================================
# 1. LOAD DATA
# ==========================================================

df = pd.read_csv('ground_truth_uhi_landsat.csv')

print("Dataset Shape:", df.shape)
print(df.head())


# ==========================================================
# 2. CALCULATE UHI INTENSITY
# UHI = LST - Mean(LST)
# ==========================================================

mean_lst = df['LST_C'].mean()
std_lst = df['LST_C'].std()

df['UHI_Intensity'] = df['LST_C'] - mean_lst

print("\nMean LST:", round(mean_lst, 2))
print("Std LST :", round(std_lst, 2))


# ==========================================================
# 3. DEFINE 5 UHI CLASSES
#
# Class 0 : Very Low UHI
# Class 1 : Low UHI
# Class 2 : Moderate UHI
# Class 3 : High UHI
# Class 4 : Very High UHI
# ==========================================================

def classify_uhi(uhi):

    if uhi < -1 * std_lst:
        return 0

    elif uhi < -0.5 * std_lst:
        return 1

    elif uhi < 0.5 * std_lst:
        return 2

    elif uhi < 1 * std_lst:
        return 3

    else:
        return 4


df['UHI_Class'] = df['UHI_Intensity'].apply(classify_uhi)


# ==========================================================
# 4. DISPLAY CLASS THRESHOLDS
# ==========================================================

print("\n===== UHI CLASS THRESHOLDS =====")

print(f"Class 0 (SUCI) : UHI < {-1*std_lst:.2f}")
print(f"Class 1 (MUCI) : {-1*std_lst:.2f} ≤ UHI < {-0.5*std_lst:.2f}")
print(f"Class 2 (TEZ)  : {-0.5*std_lst:.2f} ≤ UHI < {0.5*std_lst:.2f}")
print(f"Class 3 (MUHI) : {0.5*std_lst:.2f} ≤ UHI < {1*std_lst:.2f}")
print(f"Class 4 (SUHI) : UHI ≥ {1*std_lst:.2f}")


# ==========================================================
# 5. CLASS DISTRIBUTION
# ==========================================================

print("\n===== CLASS DISTRIBUTION =====")

class_counts = df['UHI_Class'].value_counts().sort_index()

for cls, count in class_counts.items():
    percentage = (count / len(df)) * 100
    print(f"Class {cls}: {count:,} samples ({percentage:.2f}%)")


# ==========================================================
# 6. SAVE DATASET
# ==========================================================

output_file = 'ground_truth_uhi_landsat_5class.csv'

df.to_csv(output_file, index=False)

print(f"\nDataset saved to: {output_file}")


# ==========================================================
# 7. PREVIEW
# ==========================================================

print("\n===== SAMPLE DATA =====")

print(
    df[
        [
            'LST_C',
            'UHI_Intensity',
            'UHI_Class'
        ]
    ].head(10)
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(
    "ground_truth_uhi_landsat_5class.csv",
    encoding="utf-8-sig"
)

# =========================================================
# UHI INTENSITY
# =========================================================

mean_lst = df["LST_C"].mean()

df["UHI_Intensity"] = (
    df["LST_C"] - mean_lst
)

uhi = df["UHI_Intensity"]

# =========================================================
# THRESHOLDS
# =========================================================

std_uhi = uhi.std()

t1 = -1 * std_uhi
t2 = -0.5 * std_uhi
t3 = 0.5 * std_uhi
t4 = 1.0 * std_uhi

# =========================================================
# KDE
# =========================================================

kde = gaussian_kde(uhi)

x_vals = np.linspace(
    uhi.min(),
    uhi.max(),
    500
)

y_vals = kde(x_vals)

# =========================================================
# SCALE KDE
# =========================================================

bin_count = 40

y_vals = y_vals * len(uhi) * (
    (uhi.max() - uhi.min()) / bin_count
)

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# FIGURE
# =========================================================

plt.figure(figsize=(14,6))

# =========================================================
# HISTOGRAM
# =========================================================

plt.hist(
    uhi,
    bins=bin_count,
    color="#6baed6",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.8
)

# =========================================================
# KDE CURVE
# =========================================================

plt.plot(
    x_vals,
    y_vals,
    color="red",
    linewidth=2.5,
    label="Density Curve"
)

# =========================================================
# THRESHOLDS
# =========================================================

for x in [t1, t2, t3, t4]:

    plt.axvline(
        x,
        linestyle="--",
        linewidth=2,
        color="black"
    )

# =========================================================
# LABELS
# =========================================================

ymax = y_vals.max()

plt.text(
    t1-2,
    ymax*0.90,
    "SUCI\nClass 0",
    ha="center",
    fontsize=9
)

plt.text(
    (t1+t2)/2,
    ymax*0.90,
    "MUCI\nClass 1",
    ha="center",
    fontsize=9
)

plt.text(
    0,
    ymax*0.90,
    "TEZ\nClass 2",
    ha="center",
    fontsize=9
)

plt.text(
    (t3+t4)/2,
    ymax*0.90,
    "MUHI\nClass 3",
    ha="center",
    fontsize=9
)

plt.text(
    t4+2,
    ymax*0.90,
    "SUHI\nClass 4",
    ha="center",
    fontsize=9
)

# =========================================================
# TITLE
# =========================================================

plt.title(
    "Urban Heat Island Intensity Distribution and Five-Class Classification Scheme",
    fontsize=14
)

plt.xlabel(
    "UHI Intensity (°C)",
    fontsize=12
)

plt.ylabel(
    "Frequency",
    fontsize=12
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.legend()

# =========================================================
# CLEAN STYLE
# =========================================================

ax = plt.gca()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# =========================================================
# SAVE
# =========================================================

plt.tight_layout()

plt.savefig(
    "Figure_UHI_5Class_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# =========================================================
# PRINT THRESHOLDS
# =========================================================

print("\n===== UHI CLASS THRESHOLDS =====")

print(f"SUCI : UHI < {t1:.2f}")
print(f"MUCI : {t1:.2f} ≤ UHI < {t2:.2f}")
print(f"TEZ  : {t2:.2f} ≤ UHI < {t3:.2f}")
print(f"MUHI : {t3:.2f} ≤ UHI < {t4:.2f}")
print(f"SUHI : UHI ≥ {t4:.2f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(
    "ground_truth_uhi_landsat_5class.csv"
)

# =========================================================
# CLASS LABELS
# =========================================================

class_names = {
    0: "SUCI",
    1: "MUCI",
    2: "TEZ",
    3: "MUHI",
    4: "SUHI"
}

# =========================================================
# CLASS DESCRIPTIONS
# =========================================================

class_desc = {
    0: "Strong Urban Cool Island",
    1: "Moderate Urban Cool Island",
    2: "Thermal Equilibrium Zone",
    3: "Moderate Urban Heat Island",
    4: "Strong Urban Heat Island"
}

# =========================================================
# COUNT SAMPLES
# =========================================================

counts = (
    df["UHI_Class"]
    .value_counts()
    .sort_index()
)

labels = [
    class_names[c]
    for c in counts.index
]

# =========================================================
# PERCENTAGES
# =========================================================

percentages = (
    counts / counts.sum() * 100
)

# =========================================================
# BLUE PALETTE
# =========================================================

colors = [
    "#c6dbef",  # SUCI
    "#9ecae1",  # MUCI
    "#6baed6",  # TEZ
    "#2171b5",  # MUHI
    "#08306b"   # SUHI
]

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# FIGURE
# =========================================================

plt.figure(figsize=(11, 6))

bars = plt.bar(
    labels,
    counts,
    color=colors,
    edgecolor="black",
    linewidth=0.8
)

# =========================================================
# VALUE LABELS
# =========================================================

for bar, pct in zip(bars, percentages):

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + counts.max()*0.01,
        f"{int(height):,}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

# =========================================================
# LEGEND
# =========================================================

legend_elements = [
    Patch(
        facecolor=colors[i],
        edgecolor="black",
        label=f"{class_names[i]}: {class_desc[i]}"
    )
    for i in range(5)
]

plt.legend(
    handles=legend_elements,
    fontsize=9,
    frameon=True,
    loc="upper right"
)

# =========================================================
# LABELS
# =========================================================

plt.title(
    "Distribution of Urban Heat Island Intensity Classes",
    fontsize=15,
    pad=20
)

plt.xlabel(
    "UHI Class",
    fontsize=12
)

plt.ylabel(
    "Number of Samples",
    fontsize=12
)

# =========================================================
# GRID
# =========================================================

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

# =========================================================
# CLEAN STYLE
# =========================================================

ax = plt.gca()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# =========================================================
# SAVE FIGURE
# =========================================================

plt.tight_layout()

plt.savefig(
    "Figure_UHI_Class_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# =========================================================
# PRINT SUMMARY
# =========================================================

print("\n===== UHI CLASS DISTRIBUTION =====")

for cls in sorted(counts.index):

    print(
        f"{class_names[cls]} "
        f"({class_desc[cls]}): "
        f"{counts[cls]:,} samples "
        f"({percentages[cls]:.2f}%)"
    )

In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# LOAD DATASET
# =========================================================

file_path = "/content/ground_truth_uhi_landsat_5class.csv"

df = pd.read_csv(file_path)

# =========================================================
# DISPLAY SETTINGS
# =========================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# =========================================================
# SELECT FEATURES
# =========================================================

features = [

    # Meteorological Variables
    "AMB_TEMP",
    "RH",
    "WIND_SPEED",

    # Remote Sensing Variables
    "Albedo",
    "BSI",
    "EVI",
    "MNDWI",
    "NDBI",
    "NDVI",
    "NDWI",
    "SAVI"

]

# =========================================================
# CHECK FEATURE AVAILABILITY
# =========================================================

missing_features = [
    col for col in features
    if col not in df.columns
]

if missing_features:

    print("\nMISSING FEATURE COLUMNS")
    print("=================================================")

    for col in missing_features:
        print(col)

else:

    print("\nALL SELECTED FEATURES ARE AVAILABLE")
    print("=================================================")

# =========================================================
# KEEP AVAILABLE FEATURES
# =========================================================

available_features = [
    col for col in features
    if col in df.columns
]

# =========================================================
# REMOVE MISSING VALUES
# =========================================================

df_stats = df[available_features].dropna()

print("\nDATA USED FOR STATISTICS")
print("=================================================")
print(df_stats.shape)

# =========================================================
# DESCRIPTIVE STATISTICS
# =========================================================

desc_table = (
    df_stats
    .describe()
    .T
)

# =========================================================
# ADD SKEWNESS
# =========================================================

desc_table["skewness"] = (
    df_stats
    .skew()
)

# =========================================================
# ADD MISSING VALUES
# =========================================================

desc_table["missing_values"] = (
    df[available_features]
    .isna()
    .sum()
)

# =========================================================
# SELECT IMPORTANT STATISTICS
# =========================================================

desc_table = desc_table[
    [
        "count",
        "missing_values",
        "mean",
        "std",
        "min",
        "25%",
        "50%",
        "75%",
        "max",
        "skewness"
    ]
]

# =========================================================
# RENAME COLUMNS
# =========================================================

desc_table = desc_table.rename(columns={

    "count": "Count",
    "missing_values": "Missing",
    "mean": "Mean",
    "std": "Std. Dev.",
    "min": "Min",
    "25%": "Q1",
    "50%": "Median",
    "75%": "Q3",
    "max": "Max",
    "skewness": "Skewness"

})

# =========================================================
# ROUND VALUES
# =========================================================

desc_table = desc_table.round(4)

# =========================================================
# RESET INDEX
# =========================================================

desc_table = (
    desc_table
    .reset_index()
    .rename(columns={"index": "Feature"})
)

# =========================================================
# KEEP FEATURE ORDER
# =========================================================

desc_table["Feature"] = pd.Categorical(
    desc_table["Feature"],
    categories=features,
    ordered=True
)

desc_table = desc_table.sort_values(
    by="Feature"
)

# =========================================================
# DISPLAY TABLE
# =========================================================

print("\nDESCRIPTIVE STATISTICS TABLE")
print("=================================================")

display(desc_table)

# =========================================================
# SAVE CSV
# =========================================================

save_path = "/content/UHI_Descriptive_Statistics.csv"

desc_table.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nTABLE SAVED SUCCESSFULLY")
print("=================================================")
print(save_path)

## Train-test split and class balancing

Apply stratified 80:20 train-test split and balance the training dataset using SMOTE.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(
    "/content/ground_truth_uhi_landsat_5class.csv",
    encoding="utf-8-sig"
)

# =========================================================
# FEATURES
# =========================================================

features = [

    # Meteorological Variables
    "AMB_TEMP",
    "RH",
    "WIND_SPEED",

    # Remote Sensing Variables
    "Albedo",
    "BSI",
    "EVI",
    "MNDWI",
    "NDBI",
    "NDVI",
    "NDWI",
    "SAVI"

]

# =========================================================
# TARGET
# =========================================================

target = "UHI_Class"

# =========================================================
# REMOVE NaN
# =========================================================

df_model = df.dropna(
    subset=features + [target]
).copy()

# =========================================================
# INPUTS
# =========================================================

X = df_model[features]

y = df_model[target]

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

# =========================================================
# BEFORE SMOTE
# =========================================================

print("\nCLASS DISTRIBUTION BEFORE SMOTE")
print("================================")

print(
    y_train
    .value_counts()
    .sort_index()
)

# =========================================================
# SMOTE
# =========================================================

smote = SMOTE(

    sampling_strategy="auto",

    random_state=42,

    k_neighbors=5

)

X_train_smote, y_train_smote = smote.fit_resample(

    X_train,
    y_train

)

# =========================================================
# AFTER SMOTE
# =========================================================

print("\nCLASS DISTRIBUTION AFTER SMOTE")
print("================================")

print(
    pd.Series(y_train_smote)
    .value_counts()
    .sort_index()
)

# =========================================================
# SHAPES
# =========================================================

print("\nTRAIN SHAPE AFTER SMOTE")
print("================================")

print("X_train_smote:", X_train_smote.shape)
print("y_train_smote:", y_train_smote.shape)

print("\nTEST SHAPE")
print("================================")

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# =========================================================
# TRAIN DISTRIBUTION (%)
# =========================================================

print("\nTRAIN DISTRIBUTION AFTER SMOTE (%)")
print("================================")

print(

    (
        pd.Series(y_train_smote)
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)

)

# =========================================================
# TEST DISTRIBUTION (%)
# =========================================================

print("\nTEST DISTRIBUTION (%)")
print("================================")

print(

    (
        y_test
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)

)

# =========================================================
# CLASS NAMES
# =========================================================

class_names = {

    0: "SUCI",
    1: "MUCI",
    2: "TEZ",
    3: "MUHI",
    4: "SUHI"

}

print("\nCLASS DEFINITIONS")
print("================================")

for k, v in class_names.items():
    print(f"{k} = {v}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# COUNTS
# =========================================================

train_counts = (
    y_train
    .value_counts()
    .sort_index()
)

test_counts = (
    y_test
    .value_counts()
    .sort_index()
)

train_pct = train_counts / train_counts.sum() * 100
test_pct = test_counts / test_counts.sum() * 100

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# BLUE PALETTE
# =========================================================

blue_colors = [

    "#c6dbef",  # SUCI
    "#9ecae1",  # MUCI
    "#6baed6",  # TEZ
    "#2171b5",  # MUHI
    "#08306b"   # SUHI

]

# =========================================================
# CLASS LABELS
# =========================================================

class_labels = [

    "SUCI\n(Strong Cool Island)",

    "MUCI\n(Moderate Cool Island)",

    "TEZ\n(Thermal Equilibrium)",

    "MUHI\n(Moderate Heat Island)",

    "SUHI\n(Strong Heat Island)"

]

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 7)
)

datasets = [

    (
        "Training Dataset",
        train_counts,
        train_pct
    ),

    (
        "Testing Dataset",
        test_counts,
        test_pct
    )

]

# =========================================================
# BAR PLOTS
# =========================================================

for ax, (title, counts, pct) in zip(
    axes,
    datasets
):

    bars = ax.bar(

        range(len(counts)),

        counts.values,

        color=blue_colors,

        edgecolor="black",

        linewidth=0.8

    )

    ax.set_title(
        title,
        fontsize=13,
        pad=15
    )

    ax.set_xticks(
        range(len(counts))
    )

    ax.set_xticklabels(
        class_labels,
        fontsize=9
    )

    ax.set_ylabel(
        "Number of Samples",
        fontsize=11
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    ymax = counts.max()

    for bar, c, p in zip(
        bars,
        counts.values,
        pct.values
    ):

        ax.text(

            bar.get_x() +
            bar.get_width()/2,

            bar.get_height() +
            ymax*0.015,

            f"{c:,}\n({p:.1f}%)",

            ha="center",

            va="bottom",

            fontsize=8,

            fontweight="bold"

        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# =========================================================
# MAIN TITLE
# =========================================================

plt.suptitle(

    "Class Distribution of Urban Heat Island Intensity Categories\nfor Training and Testing Datasets",

    fontsize=15,


    y=0.98

)

# =========================================================
# LAYOUT
# =========================================================

plt.tight_layout()

plt.subplots_adjust(
    top=0.85
)

# =========================================================
# SAVE
# =========================================================

plt.savefig(

    "Figure_Train_Test_UHI_Distribution.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# =========================================================
# COUNTS
# =========================================================

before_counts = (
    y_train
    .value_counts()
    .sort_index()
)

after_counts = (
    pd.Series(y_train_smote)
    .value_counts()
    .sort_index()
)

before_pct = (
    before_counts /
    before_counts.sum()
    * 100
)

after_pct = (
    after_counts /
    after_counts.sum()
    * 100
)

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# BLUE PALETTE
# =========================================================

blue_colors = [

    "#c6dbef",  # SUCI
    "#9ecae1",  # MUCI
    "#6baed6",  # TEZ
    "#2171b5",  # MUHI
    "#08306b"   # SUHI

]

# =========================================================
# CLASS LABELS
# =========================================================

class_labels = [

    "SUCI\n(Strong Cool Island)",

    "MUCI\n(Moderate Cool Island)",

    "TEZ\n(Thermal Equilibrium)",

    "MUHI\n(Moderate Heat Island)",

    "SUHI\n(Strong Heat Island)"

]

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16,7)
)

datasets = [

    (
        "Before SMOTE",
        before_counts,
        before_pct
    ),

    (
        "After SMOTE",
        after_counts,
        after_pct
    )

]

# =========================================================
# BAR PLOTS
# =========================================================

for ax, (title, counts, pct) in zip(
    axes,
    datasets
):

    bars = ax.bar(

        range(len(counts)),

        counts.values,

        color=blue_colors,

        edgecolor="black",

        linewidth=0.8

    )

    ax.set_title(
        title,
        fontsize=13
    )

    ax.set_xticks(
        range(len(counts))
    )

    ax.set_xticklabels(
        class_labels,
        fontsize=9
    )

    ax.set_ylabel(
        "Number of Samples",
        fontsize=11
    )

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    ymax = counts.max()

    for bar, c, p in zip(
        bars,
        counts.values,
        pct.values
    ):

        ax.text(

            bar.get_x() +
            bar.get_width()/2,

            bar.get_height() +
            ymax*0.015,

            f"{c:,}\n({p:.1f}%)",

            ha="center",

            va="bottom",

            fontsize=8

        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# =========================================================
# MAIN TITLE
# =========================================================

plt.suptitle(

    "Effect of SMOTE on Urban Heat Island Class Distribution",

    fontsize=15,

    y=0.98

)

# =========================================================
# LAYOUT
# =========================================================

plt.tight_layout()

plt.subplots_adjust(
    top=0.86
)

# =========================================================
# SAVE
# =========================================================

plt.savefig(

    "Figure_SMOTE_UHI_Distribution.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()